# Lab 11 MNIST and Convolutional Neural Network

A Convolutional Neural Network (CNN) is a deep learning model designed to automatically and adaptively learn spatial hierarchies of features from input images, making it highly effective for tasks like image classification and object detection by combining convolutional layers, activation functions, pooling layers, fully connected layers, and dropout


In [3]:
import torch
import torchvision.datasets as dsets
import torchvision.transforms as transforms
import torch.nn.init

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# for reproducibility
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

In [5]:
# parameters
learning_rate = 0.001
training_epochs = 15
batch_size = 100

In [ ]:
# MNIST dataset
mnist_train = dsets.MNIST(root='MNIST_data/',
                          train=True,
                          transform=transforms.ToTensor(),
                          download=True)

mnist_test = dsets.MNIST(root='MNIST_data/',
                         train=False,
                         transform=transforms.ToTensor(),
                         download=True)

In [7]:
# dataset loader
data_loader = torch.utils.data.DataLoader(dataset=mnist_train,
                                          batch_size=batch_size,
                                          shuffle=True,
                                          drop_last=True)

In [8]:
# CNN Model (2 conv layers)
class CNN(torch.nn.Module):

    def __init__(self):
        super(CNN, self).__init__()
        # L1 ImgIn shape=(?, 28, 28, 1)
        #    Conv     -> (?, 28, 28, 32)
        #    Pool     -> (?, 14, 14, 32)
        self.layer1 = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2))
        # L2 ImgIn shape=(?, 14, 14, 32)
        #    Conv      ->(?, 14, 14, 64)
        #    Pool      ->(?, 7, 7, 64)
        self.layer2 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2))
        # Final FC 7x7x64 inputs -> 10 outputs
        self.fc = torch.nn.Linear(7 * 7 * 64, 10, bias=True)
        torch.nn.init.xavier_uniform_(self.fc.weight)

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)   # Flatten them for FC
        out = self.fc(out)
        return out

In a Convolutional Neural Network (CNN), multiple layers are used to progressively extract higher-level features from the input data. Each layer has a specific role in this process.

### layer 1

**Convolution**: The layer applies 32 convolutional filters (kernels) of size 3x3 to the input image. Each filter slides over the image and performs a dot product with the local region it covers, producing 32 feature maps of shape (batch_size, 32, 28, 28)

**Pooling**: Max pooling with a 2x2 kernel and stride 2 is applied, reducing the spatial dimensions by half. The output shape becomes (batch_size, 32, 14, 14).

### layer 2

**Convolution**: Applies 64 filters to the 32 input feature maps.

**Pooling**: Reducing the spatial dimensions by half. The output shape becomes (batch_size, 64, 7, 7).

### FC

Turn the input feature map with size 7x7x64 to 10 catagories.

In [9]:
# instantiate CNN model
model = CNN().to(device)

In [10]:
# define cost/loss & optimizer
criterion = torch.nn.CrossEntropyLoss().to(device)    # Softmax
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [11]:
# train my model
total_batch = len(data_loader)
print('Learning started. It takes sometime.')
for epoch in range(training_epochs):
    avg_cost = 0

    for X, Y in data_loader:
        # image is already size of (28x28), no reshape
        # label is not one-hot encoded
        X = X.to(device)
        Y = Y.to(device)

        optimizer.zero_grad()
        hypothesis = model(X)
        cost = criterion(hypothesis, Y)
        cost.backward()
        optimizer.step()

        avg_cost += cost / total_batch

    print('[Epoch: {:>4}] cost = {:>.9}'.format(epoch + 1, avg_cost))

print('Learning Finished!')

Learning started. It takes sometime.
[Epoch:    1] cost = 0.22560814
[Epoch:    2] cost = 0.06300129
[Epoch:    3] cost = 0.0462621674
[Epoch:    4] cost = 0.0374357067
[Epoch:    5] cost = 0.0314645283
[Epoch:    6] cost = 0.0260960665
[Epoch:    7] cost = 0.0218257904
[Epoch:    8] cost = 0.0183209255
[Epoch:    9] cost = 0.0161689091
[Epoch:   10] cost = 0.013468951
[Epoch:   11] cost = 0.0102096535
[Epoch:   12] cost = 0.00997181237
[Epoch:   13] cost = 0.0081878826
[Epoch:   14] cost = 0.00624045962
[Epoch:   15] cost = 0.00679804571
Learning Finished!


In [12]:
# Test model and check accuracy
with torch.no_grad():
    X_test = mnist_test.data.view(len(mnist_test), 1, 28, 28).float().to(device)
    Y_test = mnist_test.targets.to(device)

    prediction = model(X_test)
    correct_prediction = torch.argmax(prediction, 1) == Y_test
    accuracy = correct_prediction.float().mean()
    print('Accuracy:', accuracy.item())

Accuracy: 0.9869999885559082
